# 01 · Confusores de temperatura (Tier 1)

Genera los **confusores térmicos Tier 1** para el estudio causal `NIVEL_SEQUIA_MAX → rendimiento`,
agregados a **municipio × ciclo agrícola × año** con la **misma regla de asignación año-ciclo** que
`01_limpieza_datos_lluvia` / `01_limpieza_datos_temp_min`:

| Ciclo | Meses que lo componen (año agrícola = t) |
|---|---|
| **OI** (Otoño-Invierno)  | Nov, Dic del año **t-1**  +  Ene, Feb, Mar, Abr del año **t** |
| **PV** (Primavera-Verano) | Abr, May, Jun, Jul, Ago, Sep del año **t** |

Fuente climática: **Daymet V4** (diario, 1 km) vía la **API pública Single-Pixel de ORNL**
(`https://daymet.ornl.gov/single-pixel/api/data`) — **no requiere cuenta de Google ni Earth Engine**.
Se descarga una serie diaria por punto de muestreo de cada municipio y se agrega a ciclo aquí mismo.

Confusores generados (ver discusión de DAG — todos son causa del tratamiento y del rendimiento por
vías distintas a la sequía, y no son consecuencia de la sequía del ciclo):

- `tmean_ciclo` + **anomalía** respecto a la normal municipal del ciclo (`tmean_ciclo_anom`, `_z`)
- `dias_helada` — nº de días con Tmin < 0 °C en la ventana del ciclo
- `gdd_ciclo` — grados-día de crecimiento (base/tope configurables) + anomalía (`gdd_anom`)
- `tmean_inicio` — Tª media de los 2 primeros meses del ciclo (predeterminada) + anomalía
- `tmean_preseason` — Tª media de los 2 meses previos al ciclo + anomalía
- `tmean_prev90` — **Tª media de los 90 días previos** al inicio del ciclo agrícola (condición
  térmica antecedente, ventana fija en días — no alineada a meses)
- `tmean_prev90_normal` — **promedio histórico** municipal de esos mismos 90 días por ciclo (línea base ≤ 2023)
- `tmean_prev90_anom` — **Tª previa menos referencia histórica** (`tmean_prev90 − tmean_prev90_normal`)
- `tmean_normal` — normal térmica municipal del ciclo (para interactuar con la sequía sin EF de municipio)

> **Cobertura de Daymet V4:** la API ya publica **2024 completo** (verificado). Con
> `FETCH_FIN = '2024-12-31'` se cierran OI-2024 y PV-2024; la normal municipal se mantiene con la
> línea base ≤ 2023 (`ANIO_NORMAL_MAX`). Para el año agrícola 2025 sube `ANIOS` y `FETCH_FIN`.
> Las filas con la ventana del ciclo recortada por el rango descargado quedan marcadas en
> `ventana_incompleta` (regla por cobertura real de días, no por año fijo).
>
> **Nota:** la API es *single-pixel*, así que se muestrea el/los punto(s) interior(es) del municipio,
> no la media areal del polígono. Para municipios grandes o montañosos, sube `N_PUNTOS`.

In [16]:
import io
import time
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import Point

pd.set_option('display.max_columns', 60)

# --- Parámetros --------------------------------------------------------------
PROY = Path('/Users/jaydymarchan/Desktop/causalidad')
RUTA_SHP        = PROY / 'data/01_raw/datos_lat_longitud/2024_1_00_MUN.shp'
RUTA_CACHE      = PROY / 'data/01_raw/daymet_cache'                    # 1 CSV por punto de muestreo
RUTA_DAYMET_CSV = PROY / 'data/01_raw/daymet_confusores.csv'          # tabla municipio × año × ciclo (diario -> ciclo)
RUTA_OUT        = PROY / 'data/02_processed/confusores_temp.csv'      # salida final con anomalías

ANIOS  = list(range(2016, 2025))      # años agrícolas t (igual que el merge de 02_join_data)
CICLOS = ['OI', 'PV']

GDD_TBASE = 10.0     # °C   (maíz; ajustar al cultivo dominante)
GDD_TCAP  = 30.0     # °C

DIAS_PREV = 90       # días previos al inicio del ciclo para la Tª antecedente (tmean_prev90)

N_PUNTOS = 1                          # puntos Daymet por municipio (interior); 3-5 para municipios grandes
PAUSA    = 0.3                        # segundos entre descargas (cortesía con el servidor ORNL)
# Ventana de descarga Daymet V4:
#  - FETCH_INI en agosto para cubrir los 90 días previos de OI-2016 (el ciclo arranca 2015-11-01).
#  - FETCH_FIN llega a fin de 2024 para cerrar OI-2024 (Nov23-Abr24) y PV-2024 (Abr24-Sep24).
#    Daymet V4 ya publica 2024 completo (verificado). Sube a '2025-12-31' si añades el año agrícola 2025.
# Si el caché en data/01_raw/daymet_cache es anterior a este cambio, BÓRRALO para re-descargar el rango nuevo
# (la celda de descarga se salta los archivos que ya existen, no revalida las fechas).
FETCH_INI, FETCH_FIN = '2015-08-01', '2024-12-31'

## 1 · Puntos de muestreo por municipio

In [17]:
gdf = gpd.read_file(RUTA_SHP)[['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'NOMGEO', 'geometry']].to_crs('EPSG:4326')
gdf['idestado']    = gdf['CVE_ENT'].astype(int)
gdf['idmunicipio'] = gdf['CVE_MUN'].astype(int)


def puntos_muestreo(geom, n):
    """1..n puntos garantizados dentro del polígono (representativo + rejilla recortada)."""
    pts = [geom.representative_point()]
    if n > 1:
        minx, miny, maxx, maxy = geom.bounds
        g = max(2, int(np.ceil(np.sqrt(n * 4))))
        cand = [Point(x, y)
                for x in np.linspace(minx, maxx, g + 2)[1:-1]
                for y in np.linspace(miny, maxy, g + 2)[1:-1]]
        pts += [p for p in cand if geom.contains(p)]
    return pts[:n]


filas = []
for r in gdf.itertuples():
    for k, p in enumerate(puntos_muestreo(r.geometry, N_PUNTOS)):
        filas.append((r.CVEGEO, r.idestado, r.idmunicipio, r.NOMGEO, k,
                      round(p.y, 5), round(p.x, 5)))
pts = pd.DataFrame(filas, columns=['CVEGEO', 'idestado', 'idmunicipio', 'NOMGEO', 'punto', 'lat', 'lon'])
print(f'{len(gdf)} municipios -> {len(pts)} puntos de muestreo (N_PUNTOS={N_PUNTOS})')
pts.head()

2478 municipios -> 2478 puntos de muestreo (N_PUNTOS=1)


,CVEGEO,idestado,idmunicipio,NOMGEO,punto,lat,lon
0,01005,1,5,Jesús María,0,21.92872,-102.46152
1,01008,1,8,San José de Gracia,0,22.15372,-102.52423
2,01001,1,1,Aguascalientes,0,21.84859,-102.30550
3,01004,1,4,Cosío,0,22.37052,-102.31201
4,01011,1,11,San Francisco de los Romo,0,22.02165,-102.25180


## 2 · Regla de asignación año–ciclo

In [18]:
# Meses (calendario) que componen cada ciclo: (desfase_año, mes)  ->  desfase 0 = año t, -1 = año t-1
_MESES_CICLO = {
    'OI': [(-1, 11), (-1, 12), (0, 1), (0, 2), (0, 3), (0, 4)],
    'PV': [(0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9)],
}


def _rango(anio, meses):
    """Lista contigua de (desfase, mes) -> (inicio, fin_exclusiva) en 'YYYY-MM-DD'."""
    y0, m0 = anio + meses[0][0],  meses[0][1]
    yN, mN = anio + meses[-1][0], meses[-1][1]
    ini = f'{y0:04d}-{m0:02d}-01'
    fin = f'{yN + mN // 12:04d}-{mN % 12 + 1:02d}-01'          # 1er día del mes siguiente al último
    return ini, fin


def ventana_ciclo(anio, ciclo, parte='completo'):
    """
    Regla idéntica a 01_limpieza_datos_lluvia / 01_limpieza_datos_temp_min:

        OI(t) = Nov, Dic (t-1)  +  Ene, Feb, Mar, Abr (t)
        PV(t) = Abr, May, Jun, Jul, Ago, Sep (t)

    parte:
        'completo'   -> toda la ventana del ciclo
        'inicio'     -> los 2 primeros meses del ciclo (temperatura predeterminada)
        'preseason'  -> los 2 meses previos al primer mes del ciclo
    Devuelve (inicio, fin_exclusiva) como 'YYYY-MM-DD'.
    """
    meses = _MESES_CICLO[ciclo]
    if parte == 'completo':
        return _rango(anio, meses)
    if parte == 'inicio':
        return _rango(anio, meses[:2])
    if parte == 'preseason':
        o0, m0 = meses[0]
        prev = [(o0 + (m0 - 2 - 1) // 12, (m0 - 2 - 1) % 12 + 1),
                (o0 + (m0 - 1 - 1) // 12, (m0 - 1 - 1) % 12 + 1)]
        return _rango(anio, prev)
    raise ValueError(parte)


def ventana_prev(anio, ciclo, dias=DIAS_PREV):
    """
    Ventana de los `dias` días **inmediatamente previos** al primer día del ciclo agrícola
    (condición térmica antecedente). A diferencia de 'preseason', es una ventana fija en días,
    no alineada a meses:

        OI(t) -> [inicio_OI - dias, inicio_OI),  con inicio_OI = 1-nov de t-1
        PV(t) -> [inicio_PV - dias, inicio_PV),  con inicio_PV = 1-abr de t

    Devuelve (inicio, fin_exclusiva) como pd.Timestamp.
    """
    ini_ciclo, _ = ventana_ciclo(anio, ciclo, 'completo')
    fin = pd.Timestamp(ini_ciclo)
    return fin - pd.Timedelta(days=dias), fin


for c in CICLOS:
    for p in ('completo', 'inicio', 'preseason'):
        print(f'{c} 2020 {p:11s}', ventana_ciclo(2020, c, p))
    i0, i1 = ventana_prev(2020, c, DIAS_PREV)
    print(f'{c} 2020 prev{DIAS_PREV:<7d}', (i0.date().isoformat(), i1.date().isoformat()))

OI 2020 completo    ('2019-11-01', '2020-05-01')
OI 2020 inicio      ('2019-11-01', '2020-01-01')
OI 2020 preseason   ('2019-09-01', '2019-11-01')
OI 2020 prev90      ('2019-08-03', '2019-11-01')
PV 2020 completo    ('2020-04-01', '2020-10-01')
PV 2020 inicio      ('2020-04-01', '2020-06-01')
PV 2020 preseason   ('2020-02-01', '2020-04-01')
PV 2020 prev90      ('2020-01-02', '2020-04-01')


## 3 · Descarga Daymet (API Single-Pixel, sin Google)

~2 478 descargas (una por punto), pequeñas y **reanudables**: cada CSV se guarda en `RUTA_CACHE` y
al re-ejecutar la celda se saltan los que ya están. Con `PAUSA=0.3` tarda ~15 min.

In [19]:
RUTA_CACHE.mkdir(parents=True, exist_ok=True)
URL = 'https://daymet.ornl.gov/single-pixel/api/data'


def _descarga(lat, lon, destino):
    if destino.exists() and destino.stat().st_size > 500:
        return 'cache'
    for intento in range(3):
        try:
            resp = requests.get(URL, params={'lat': lat, 'lon': lon, 'vars': 'tmax,tmin',
                                             'start': FETCH_INI, 'end': FETCH_FIN}, timeout=90)
            if resp.status_code == 200 and 'year,yday' in resp.text:
                destino.write_text(resp.text)
                return 'ok'
        except requests.RequestException:
            pass
        time.sleep(2 * (intento + 1))
    return 'error'


ok = cache = 0
errores = []
for i, r in enumerate(pts.itertuples(), 1):
    res = _descarga(r.lat, r.lon, RUTA_CACHE / f'{r.CVEGEO}_{r.punto}.csv')
    if res == 'ok':
        ok += 1
        time.sleep(PAUSA)
    elif res == 'cache':
        cache += 1
    else:
        errores.append(f'{r.CVEGEO}_{r.punto}')
    if i % 200 == 0:
        print(f'{i}/{len(pts)}  ok={ok} cache={cache} error={len(errores)}')

print(f'\nlisto: {ok} descargados, {cache} en cache, {len(errores)} errores')
if errores:
    print('con error (re-ejecuta la celda para reintentar):', errores[:20])

200/2478  ok=199 cache=0 error=1
400/2478  ok=399 cache=0 error=1
600/2478  ok=599 cache=0 error=1
800/2478  ok=799 cache=0 error=1
1000/2478  ok=999 cache=0 error=1
1200/2478  ok=1199 cache=0 error=1
1400/2478  ok=1398 cache=0 error=2
1600/2478  ok=1598 cache=0 error=2
1800/2478  ok=1798 cache=0 error=2
2000/2478  ok=1997 cache=0 error=3
2200/2478  ok=2197 cache=0 error=3
2400/2478  ok=2396 cache=0 error=4

listo: 2474 descargados, 0 en cache, 4 errores
con error (re-ejecuta la celda para reintentar): ['07016_0', '20043_0', '25009_0', '30133_0']


## 4 · Serie diaria → agregados por ciclo

In [20]:
def _leer(path):
    lin = path.read_text().splitlines()
    j = next(k for k, l in enumerate(lin) if l.startswith('year,yday'))
    d = pd.read_csv(io.StringIO('\n'.join(lin[j:])))
    d.columns = ['year', 'yday', 'tmax', 'tmin']
    d['fecha'] = pd.to_datetime(d['year'].astype(str) + '-01-01') + pd.to_timedelta(d['yday'] - 1, unit='D')
    d['tmean'] = (d['tmax'] + d['tmin']) / 2
    d['helada'] = (d['tmin'] < 0).astype(int)
    d['gdd'] = d['tmean'].clip(lower=GDD_TBASE, upper=GDD_TCAP) - GDD_TBASE
    return d[['fecha', 'tmin', 'tmax', 'tmean', 'helada', 'gdd']]


# serie diaria de cada municipio = promedio de sus N_PUNTOS
diarios = {}
for (cve, ide, idm, nom), grp in pts.groupby(['CVEGEO', 'idestado', 'idmunicipio', 'NOMGEO']):
    partes = [_leer(RUTA_CACHE / f'{r.CVEGEO}_{r.punto}.csv')
              for r in grp.itertuples() if (RUTA_CACHE / f'{r.CVEGEO}_{r.punto}.csv').exists()]
    if partes:
        diarios[(ide, idm, cve, nom)] = (
            pd.concat(partes).groupby('fecha', as_index=False).mean(numeric_only=True)
        )
print(f'{len(diarios)} municipios con serie diaria Daymet')


def _media(dd, ini, fin, col):
    m = dd.loc[(dd['fecha'] >= pd.Timestamp(ini)) & (dd['fecha'] < pd.Timestamp(fin)), col]
    return m.mean() if len(m) else np.nan


def _cobertura(dd, ini, fin):
    """nº de días con dato en [ini, fin) — para detectar ventanas recortadas por el rango de Daymet."""
    m = dd.loc[(dd['fecha'] >= pd.Timestamp(ini)) & (dd['fecha'] < pd.Timestamp(fin)), 'tmean']
    return int(m.notna().sum())


registros = []
for (ide, idm, cve, nom), dd in diarios.items():
    for a in ANIOS:
        for c in CICLOS:
            ini, fin = ventana_ciclo(a, c, 'completo')
            ii, fi = ventana_ciclo(a, c, 'inicio')
            ip, fp = ventana_ciclo(a, c, 'preseason')
            ipp, fpp = ventana_prev(a, c, DIAS_PREV)          # 90 días previos al inicio del ciclo
            w = dd[(dd['fecha'] >= pd.Timestamp(ini)) & (dd['fecha'] < pd.Timestamp(fin))]
            registros.append(dict(
                idestado=ide, idmunicipio=idm, CVEGEO=cve, NOMGEO=nom,
                anio=a, nomcicloproductivo=c, ventana_ini=ini, ventana_fin=fin,
                tmin_ciclo_mean=w['tmin'].mean() if len(w) else np.nan,
                tmax_ciclo_mean=w['tmax'].mean() if len(w) else np.nan,
                tmean_ciclo=w['tmean'].mean() if len(w) else np.nan,
                dias_helada=w['helada'].sum() if len(w) else np.nan,
                gdd_ciclo=w['gdd'].sum() if len(w) else np.nan,
                tmean_inicio=_media(dd, ii, fi, 'tmean'),
                tmean_preseason=_media(dd, ip, fp, 'tmean'),
                tmean_prev90=_media(dd, ipp, fpp, 'tmean'),
                n_dias_con_dato=int(w['tmean'].notna().sum()),     # días Daymet dentro de la ventana del ciclo
                n_dias_prev90=_cobertura(dd, ipp, fpp),
            ))

raw = pd.DataFrame(registros)
raw.to_csv(RUTA_DAYMET_CSV, index=False, encoding='utf-8')
print(raw.shape, '->', RUTA_DAYMET_CSV)
raw.head()

2474 municipios con serie diaria Daymet
(44532, 18) -> /Users/jaydymarchan/Desktop/causalidad/data/01_raw/daymet_confusores.csv


,idestado,idmunicipio,CVEGEO,NOMGEO,anio,nomcicloproductivo,ventana_ini,ventana_fin,tmin_ciclo_mean,tmax_ciclo_mean,tmean_ciclo,dias_helada,gdd_ciclo,tmean_inicio,tmean_preseason,tmean_prev90,n_dias_con_dato,n_dias_prev90
0,1,1,01001,Aguascalientes,2016,OI,2015-11-01,2016-05-01,8.388626,24.912967,16.650797,1.0,1217.63,16.305738,20.320328,20.659500,182,90
1,1,1,01001,Aguascalientes,2016,PV,2016-04-01,2016-10-01,14.091694,29.001858,21.546776,0.0,2113.06,21.827541,16.758167,15.601389,183,90
2,1,1,01001,Aguascalientes,2017,OI,2016-11-01,2017-05-01,8.665500,25.948167,17.306833,0.0,1315.23,16.441000,20.032623,20.373278,180,90
3,1,1,01001,Aguascalientes,2017,PV,2017-04-01,2017-10-01,14.029945,29.392678,21.711311,0.0,2143.17,21.856393,17.722712,16.827333,183,90
4,1,1,01001,Aguascalientes,2018,OI,2017-11-01,2018-05-01,8.450387,26.021271,17.235829,0.0,1311.62,15.966721,19.563852,20.224889,181,90


## 5 · Anomalías respecto a la normal municipal del ciclo

In [21]:
raw = pd.read_csv(RUTA_DAYMET_CSV)
raw['idestado']    = raw['idestado'].astype(int)
raw['idmunicipio'] = raw['idmunicipio'].astype(int)
raw['anio']        = raw['anio'].astype(int)

LLAVE = ['idestado', 'idmunicipio', 'nomcicloproductivo']
NIV   = ['tmean_ciclo', 'gdd_ciclo', 'tmean_inicio', 'tmean_preseason', 'tmean_prev90']

ANIO_NORMAL_MAX = 2023          # años usados para la normal municipal por ciclo (línea base)

# Normal municipal por ciclo con los años base de Daymet
base = raw[raw['anio'] <= ANIO_NORMAL_MAX]
norm = base.groupby(LLAVE)[NIV].agg(['mean', 'std'])
norm.columns = [f'{c}_{s}' for c, s in norm.columns]
norm = norm.reset_index()

df = raw.merge(norm, on=LLAVE, how='left')

for c in NIV:
    df[f'{c}_anom'] = df[c] - df[f'{c}_mean']
df['tmean_ciclo_anom_z'] = (
    (df['tmean_ciclo'] - df['tmean_ciclo_mean']) / df['tmean_ciclo_std'].replace(0, np.nan)
)


def _n_dias(r):
    y0, m0, d0 = map(int, r['ventana_ini'].split('-'))
    y1, m1, d1 = map(int, r['ventana_fin'].split('-'))
    return (date(y1, m1, d1) - date(y0, m0, d0)).days


df['n_dias_ventana'] = df.apply(_n_dias, axis=1)
# incompleta = sin dato, o con la ventana recortada por el rango descargado de Daymet
# (umbral 0.95 porque Daymet entrega 365 días/año -> pierde el 31-dic de los años bisiestos).
df['ventana_incompleta'] = df['tmean_ciclo'].isna() | (df['n_dias_con_dato'] < 0.95 * df['n_dias_ventana'])
df['prev90_incompleta']  = df['tmean_prev90'].isna() | (df['n_dias_prev90'] < 0.9 * DIAS_PREV)

df_confusores = (
    df[['idestado', 'idmunicipio', 'nomcicloproductivo', 'anio',
        'tmean_ciclo', 'tmin_ciclo_mean', 'tmax_ciclo_mean', 'dias_helada', 'gdd_ciclo',
        'tmean_inicio', 'tmean_preseason',
        'tmean_prev90', 'tmean_prev90_mean', 'tmean_prev90_anom',
        'tmean_ciclo_mean', 'tmean_ciclo_anom', 'tmean_ciclo_anom_z',
        'gdd_ciclo_mean', 'gdd_ciclo_anom',
        'tmean_inicio_anom', 'tmean_preseason_anom',
        'n_dias_ventana', 'n_dias_con_dato', 'n_dias_prev90',
        'ventana_incompleta', 'prev90_incompleta']]
    .rename(columns={'tmean_ciclo_mean': 'tmean_normal',
                     'gdd_ciclo_mean': 'gdd_normal',
                     'gdd_ciclo_anom': 'gdd_anom',
                     'tmean_prev90_mean': 'tmean_prev90_normal'})
    .sort_values(['idestado', 'idmunicipio', 'anio', 'nomcicloproductivo'])
    .reset_index(drop=True)
)
df_confusores.head(12)

,idestado,idmunicipio,nomcicloproductivo,anio,tmean_ciclo,tmin_ciclo_mean,tmax_ciclo_mean,dias_helada,gdd_ciclo,tmean_inicio,tmean_preseason,tmean_prev90,tmean_prev90_normal,tmean_prev90_anom,tmean_normal,tmean_ciclo_anom,tmean_ciclo_anom_z,gdd_normal,gdd_anom,tmean_inicio_anom,tmean_preseason_anom,n_dias_ventana,n_dias_con_dato,n_dias_prev90,ventana_incompleta,prev90_incompleta
0,1,1,OI,2016,16.650797,8.388626,24.912967,1.0,1217.630,16.305738,20.320328,20.659500,20.432361,0.227139,16.917764,-0.266968,-0.678835,1261.85625,-44.22625,0.357241,0.371158,182,182,90,False,False
1,1,1,PV,2016,21.546776,14.091694,29.001858,0.0,2113.060,21.827541,16.758167,15.601389,16.496674,-0.895285,21.587001,-0.040225,-0.137596,2120.58125,-7.52125,0.609252,-0.767370,183,183,90,False,False
2,1,1,OI,2017,17.306833,8.665500,25.948167,0.0,1315.230,16.441000,20.032623,20.373278,20.432361,-0.059083,16.917764,0.389069,0.989310,1261.85625,53.37375,0.492503,0.083453,181,180,90,False,False
3,1,1,PV,2017,21.711311,14.029945,29.392678,0.0,2143.170,21.856393,17.722712,16.827333,16.496674,0.330660,21.587001,0.124310,0.425218,2120.58125,22.58875,0.638105,0.197175,183,183,90,False,False
4,1,1,OI,2018,17.235829,8.450387,26.021271,0.0,1311.620,15.966721,19.563852,20.224889,20.432361,-0.207472,16.917764,0.318064,0.808762,1261.85625,49.76375,0.018225,-0.385318,181,181,90,False,False
5,1,1,PV,2018,21.338169,13.803880,28.872459,0.0,2074.885,21.741148,18.526017,16.932444,16.496674,0.435771,21.587001,-0.248832,-0.851161,2120.58125,-45.69625,0.522859,1.000481,183,183,90,False,False
6,1,1,OI,2019,16.550028,8.017901,25.082155,1.0,1210.510,14.272049,19.657377,20.067000,20.432361,-0.365361,16.917764,-0.367737,-0.935067,1261.85625,-51.34625,-1.676447,-0.291793,181,181,90,False,False
7,1,1,PV,2019,21.609781,13.893716,29.325847,0.0,2124.590,21.020000,18.183644,16.995889,16.496674,0.499215,21.587001,0.022780,0.077922,2120.58125,4.00875,-0.198289,0.658108,183,183,90,False,False
8,1,1,OI,2020,17.421951,8.635220,26.208681,0.0,1350.795,16.465246,20.525984,21.080722,20.432361,0.648361,16.917764,0.504186,1.282025,1261.85625,88.93875,0.516749,0.576814,182,182,90,False,False
9,1,1,PV,2020,21.826585,13.732131,29.921038,0.0,2164.265,21.771230,17.978667,16.802833,16.496674,0.306160,21.587001,0.239583,0.819525,2120.58125,43.68375,0.552941,0.453130,183,183,90,False,False


In [22]:
df_confusores.to_csv(RUTA_OUT, index=False, encoding='utf-8')
print(df_confusores.shape, '->', RUTA_OUT)

(44532, 26) -> /Users/jaydymarchan/Desktop/causalidad/data/02_processed/confusores_temp.csv


## 6 · Sanity check

In [23]:
esperado = len(ANIOS) * len(CICLOS)                      # 18 combinaciones año-ciclo por municipio
cob = (df_confusores.groupby(['idestado', 'idmunicipio'])
       .agg(n=('anio', 'size'),
            n_completos=('ventana_incompleta', lambda s: int((~s).sum())))
       .reset_index())

print(f'municipios: {len(cob)}  |  combinaciones año-ciclo esperadas por municipio: {esperado}')
print(f'municipios con las {esperado} filas: {(cob["n"] == esperado).sum()}')
print(f'municipios con panel Daymet completo ({esperado} ventanas): {(cob["n_completos"] == esperado).sum()}')

print('\n% filas con ventana del ciclo incompleta por año x ciclo:')
print(df_confusores.pivot_table(index='anio', columns='nomcicloproductivo',
                                values='ventana_incompleta', aggfunc='mean').round(3))

print('\n% filas con prev90 incompleto por año x ciclo (esperado solo en OI-2016 si el caché no cubre ago-2015):')
print(df_confusores.pivot_table(index='anio', columns='nomcicloproductivo',
                                values='prev90_incompleta', aggfunc='mean').round(3))

print('\ndescripción de los confusores Tier 1:')
print(df_confusores[['tmean_ciclo', 'tmean_ciclo_anom', 'tmean_ciclo_anom_z',
                     'dias_helada', 'gdd_ciclo', 'gdd_anom',
                     'tmean_inicio_anom', 'tmean_preseason_anom',
                     'tmean_prev90', 'tmean_prev90_normal', 'tmean_prev90_anom']].describe().round(2))

municipios: 2474  |  combinaciones año-ciclo esperadas por municipio: 18
municipios con las 18 filas: 2474
municipios con panel Daymet completo (18 ventanas): 2474

% filas con ventana del ciclo incompleta por año x ciclo:
nomcicloproductivo   OI   PV
anio                        
2016                0.0  0.0
2017                0.0  0.0
2018                0.0  0.0
2019                0.0  0.0
2020                0.0  0.0
2021                0.0  0.0
2022                0.0  0.0
2023                0.0  0.0
2024                0.0  0.0

% filas con prev90 incompleto por año x ciclo (esperado solo en OI-2016 si el caché no cubre ago-2015):
nomcicloproductivo   OI   PV
anio                        
2016                0.0  0.0
2017                0.0  0.0
2018                0.0  0.0
2019                0.0  0.0
2020                0.0  0.0
2021                0.0  0.0
2022                0.0  0.0
2023                0.0  0.0
2024                0.0  0.0

descripción de los confusores Tie

## 7 · Uso en el modelo

`confusores_temp.csv` se une a `df_join` (`02_join_data`) por las 4 llaves:

```python
df_join = df_join.merge(
    pd.read_csv('data/02_processed/confusores_temp.csv'),
    on=['idestado', 'idmunicipio', 'anio', 'nomcicloproductivo'],
    how='left',
)
```

Controles Tier 1 sugeridos:

- `tmean_ciclo_anom` (o `tmean_ciclo_anom_z`) — nivel térmico del ciclo, desviado de la normal municipal
- `dias_helada` — daño por helada, independiente de la sequía
- `gdd_anom` — desarrollo del cultivo
- `tmean_inicio_anom` — condición térmica predeterminada al arranque del ciclo
- `tmean_prev90_anom` — **Tª de los 90 días previos al ciclo menos su referencia histórica**;
  captura el estado térmico antecedente (suelo/humedad de arranque) sin ser posterior al tratamiento.
  Usa `tmean_prev90` crudo solo con EF de municipio; con panel sin EF, prefiere la anomalía.
- `tmean_normal` — para **interactuar** con `NIVEL_SEQUIA_MAX` si el modelo no lleva EF de municipio

Columnas de apoyo (no meter como control): `tmean_prev90`, `tmean_prev90_normal`, `n_dias_prev90`,
`prev90_incompleta` (marca las filas cuya ventana de 90 días quedó recortada por el inicio de Daymet).

Deja fuera (Tier 2/3): Tmax media del ciclo, EDD/VPD, y cualquier variable posterior al tratamiento.